# Peer-to-Peer (P2P) Access

## Aside: Unified Virtual Addressing (UVA)

Without Unified Virtual Addressing (UVA), each GPU and the CPU maintain their own separate address space.
Pointer values, i.e. addresses, need additional information (their allocation device).

<table>
  <tr>
    <td align="center">
        <img src="./img/uva-without.png" width="640" alt="UVA Visualization (Without)"/>
    </td>
</div>

UVA merges all host and GPU memories into a single, unified **virtual** address space.
As a consequence, pointers are now unique across the system, and the device whose memory they are pointing to can be inferred from the pointer value itself.

<table>
  <tr>
    <td align="center">
        <img src="./img/uva-with.png" width="640" alt="UVA Visualization"/>
    </td>
</div>

UVA simplifies multi-GPU and heterogeneous memory programming by enabling direct pointer-based access and streamlined data transfer.
Features such as GPU access to pinned host memory and peer-to-peer GPU communication depend on UVA.
However, UVA does not automatically make all memory universally accessible:
Hardware support is still necessary, and peer access capabilities (see below) need to be enabled.

## P2P Access

Next we optimize the halo exchanges which are currently implemented using `cudaMemcpyAsync(..., cudaMemcpyDeviceToDevice)`.
Without **peer-to-peer access (P2P)**, which is the default, such transfers are routed through system memory (host-mediated), causing extra latency and PCIe bandwidth waste.
By enabling P2P, compatible GPUs on the same PCIe root complex or NVLink fabric can perform **direct device-to-device copies**, i.e. multiple GPUs can directly read and write each other's memory without staging data through host memory.

<table>
  <tr>
    <td align="center">
      <img src="./img/p2p-without.png" height="320" style="background-color:white" alt="Data Routing Visualization"><br>
      <div>Without GPUDirect, P2P data transfers have to be relayed via the host.</div>
    </td>
    <td align="center">
      <img src="./img/p2p-nvlink.png" height="320" style="background-color:white" alt="Data Routing Visualization"><br>
      <div>With GPUDirect P2P, data transfers can be done directly via NVLink...</div>
    </td>
    <td align="center">
      <img src="./img/p2p-pci.png" height="320" style="background-color:white" alt="Data Routing Visualization"><br>
      <div>... or other connections if GPUs share part of their PCIe connections.</div>
    </td>
  </tr>
</table>

### 1. Query and Enable Peer Access

P2P access is an opt-in feature.
To enable it, `cudaDeviceEnablePeerAccess()` can be used.
It's good practice to first check whether P2P is available, which can be done with `cudaDeviceCanAccessPeer()` that tests if a given device pair shares a P2P path (e.g., NVLink, same PCIe switch).

After enabling P2P access, transfers between GPUs will transparently use the direct link without further steps necessary.

Note that each device must enable access to all peers it will exchange data with.
A robust and straight-forward approach is enabling access on an all-to-all basis.

```cpp
for (int srcDeviceIdx = 0; srcDeviceIdx < numDevices; ++srcDeviceIdx) {
    checkCudaError(cudaSetDevice(srcDeviceIdx));
    for (int dstDeviceIdx = 0; dstDeviceIdx < numDevices; ++dstDeviceIdx) {
        if (srcDeviceIdx != dstDeviceIdx) {
            int canAccessPeer = 0;
            checkCudaError(cudaDeviceCanAccessPeer(&canAccessPeer, srcDeviceIdx, dstDeviceIdx));
            if (canAccessPeer)
                checkCudaError(cudaDeviceEnablePeerAccess(dstDeviceIdx, 0));
        }
    }
}
```


In the present stencil example, only neighboring patches, and thereby GPUs, need to exchange data.
An updated code looks like this:

```cpp
for (int srcDeviceIdx = 0; srcDeviceIdx < numDevices; ++srcDeviceIdx) {
    checkCudaError(cudaSetDevice(srcDeviceIdx));

    for (int dstDeviceIdx = srcDeviceIdx - 1; dstDeviceIdx < numDevices; dstDeviceIdx += 2) {
        if (dstDeviceIdx >= 0) {
            int canAccessPeer = 0;
            checkCudaError(cudaDeviceCanAccessPeer(&canAccessPeer, srcDeviceIdx, dstDeviceIdx));
            if (canAccessPeer)
                checkCudaError(cudaDeviceEnablePeerAccess(dstDeviceIdx, 0));
        }
    }
}
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/08-p2p/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/08-p2p/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/08-p2p/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/08-p2p/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/08-p2p/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/08-p2p/stencil-2d-medium.cu ../src/08-p2p/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/08-p2p/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/08-p2p/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/08-p2p/stencil-2d-easier.cu ../src/08-p2p/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/08-p2p/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/08-p2p/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/08-p2p/stencil-2d-solution.cu ../src/08-p2p/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [08-p2p/stencil-2d.cu](../src/08-p2p/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/08-p2p ../src/08-p2p/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/08-p2p 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/08-p2p $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/08-p2p.out --error=../output/08-p2p.err \
    --wrap="../build/08-p2p $((32 * 1024)) 256 2 8 0"

cat ../output/08-p2p.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs in the `--gres=gpu:a100:NGPU` to anything between one and eight GPUs.

The profile is then available at [profiles/08-p2p.nsys-rep](../profiles/08-p2p.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/08-p2p-nsys.out --error=../output/08-p2p-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/08-p2p \
        ../build/08-p2p $((32 * 1024)) 256 2 8 0"

cat ../output/08-p2p-nsys.out

## Next Step

The next step is optimizing the application by overlapping data transfers and computation in [09](./09-overlap.ipynb).